In [5]:
# import libraries
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime

In [6]:
# define crypto assets
crypto_tickers = {
    "BTC": "BTC-USD",
    "ETH": "ETH-USD",
    "BNB": "BNB-USD",
    "SOL": "SOL-USD",
    "ADA": "ADA-USD"
}

In [7]:
# define time range
start_date = "2020-01-01"
end_date = datetime.today().strftime("%Y-%m-%d")

In [8]:
# download the data
all_data = []

for coin, ticker in crypto_tickers.items():
    print(f"Downloading {coin} ({ticker})...")
    
    try:
        df = yf.download(
            ticker,
            start=start_date,
            end=end_date,
            progress=False,
            auto_adjust=False,
            threads=False
        )
        
        if df.empty:
            print(f"Skipped {coin}: empty dataframe")
            continue

        # if yfinance returns multi-index columns, flatten them
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df = df.reset_index()

        # keep only needed columns
        expected_cols = ["Date", "Open", "High", "Low", "Close", "Volume"]
        df = df[[col for col in expected_cols if col in df.columns]]
         # add coin label
        df["coin"] = coin

        # reorder columns
        df = df[["Date", "coin", "Open", "High", "Low", "Close", "Volume"]]

        all_data.append(df)
        print(f"{coin}: {len(df)} rows")

    except Exception as e:
        print(f"Failed for {coin}: {e}")

crypto_df = pd.concat(all_data, ignore_index=True)

BTC: 2265 rows
ETH: 2265 rows
BNB: 2265 rows
SOL: 2165 rows
ADA: 2265 rows


In [9]:
crypto_df.head()

Price,Date,coin,Open,High,Low,Close,Volume
0,2020-01-01,BTC,7194.892090,7254.330566,7174.944336,7200.174316,18565664997
1,2020-01-02,BTC,7202.551270,7212.155273,6935.270020,6985.470215,20802083465
2,2020-01-03,BTC,6984.428711,7413.715332,6914.996094,7344.884277,28111481032
3,2020-01-04,BTC,7345.375488,7427.385742,7309.514160,7410.656738,18444271275
4,2020-01-05,BTC,7410.451660,7544.497070,7400.535645,7411.317383,19725074095


In [10]:
crypto_df.columns

Index(['Date', 'coin', 'Open', 'High', 'Low', 'Close', 'Volume'], dtype='object', name='Price')

In [11]:
crypto_df.groupby("coin").size()

coin
ADA    2265
BNB    2265
BTC    2265
ETH    2265
SOL    2165
dtype: int64

In [12]:
crypto_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11225 entries, 0 to 11224
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    11225 non-null  datetime64[ns]
 1   coin    11225 non-null  object        
 2   Open    11225 non-null  float64       
 3   High    11225 non-null  float64       
 4   Low     11225 non-null  float64       
 5   Close   11220 non-null  float64       
 6   Volume  11225 non-null  int64         
dtypes: datetime64[ns](1), float64(4), int64(1), object(1)
memory usage: 614.0+ KB


In [13]:
crypto_df.describe()

Price,Date,Open,High,Low,Close,Volume
count,11225,11225.000000,11225.000000,11225.000000,11220.000000,1.122500e+04
mean,2023-02-15 15:26:51.581291776,10253.618605,10457.687346,10036.129941,10257.135389,1.226361e+10
min,2020-01-01 00:00:00,0.023954,0.025993,0.019130,0.023961,6.520200e+05
25%,2021-08-04 00:00:00,16.878132,17.361557,16.368177,16.887236,7.587159e+08
50%,2023-02-16 00:00:00,307.752533,314.276611,301.588654,307.958603,3.004670e+09
75%,2024-08-30 00:00:00,3104.334717,3182.528320,3032.603271,3104.954773,1.829616e+10
max,2026-03-14 00:00:00,124752.140625,126198.070312,123196.046875,124752.531250,3.509679e+11
std,NaN,23746.324983,24171.287906,23289.100550,23750.211388,1.796959e+10


In [14]:
crypto_df.to_csv("../data/raw/crypto_prices_raw.csv", index=False)